# MACURA-drone — environment demo & training
Create the PyBullet quadrotor env, show some drone scenarios as photos, then
train the four algorithms (MACURA, MBPO, M2AC, SAC).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, warnings
os.environ['MUJOCO_GL'] = 'egl'
warnings.filterwarnings('ignore')
import torch
assert torch.cuda.is_available(), 'Enable GPU: Runtime > Change runtime type > GPU.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
TOKEN = None
try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None
if not TOKEN:
    import getpass
    TOKEN = getpass.getpass('GitHub token (input hidden): ')
os.environ['GITHUB_TOKEN'] = TOKEN
print('token loaded')

In [ ]:
%%bash
set -e
curl -sH "Authorization: token ${GITHUB_TOKEN}" \
  "https://raw.githubusercontent.com/silvergjeka22/macura-drone/main/bash/setup_colab.sh" \
  -o /content/setup_colab.sh
bash /content/setup_colab.sh

In [ ]:
import sys
if '/content/macura-drone' not in sys.path:
    sys.path.insert(0, '/content/macura-drone')
import copy, json, yaml, numpy as np
import gymnasium
gymnasium.logger.set_level(40)   # silence WARN (Box precision, etc.)
from envs import drone_env
from viz import plots
from training import train as trainer

CFG = yaml.safe_load(open('/content/macura-drone/configs/macura_drone.yaml'))
DRIVE = CFG['experiment']['drive_root']
SMOKE = True   # tiny run for a quick pass; False = full matrix

def make_cfg(smoke=False, fast=True):
    c = copy.deepcopy(CFG)
    if fast:
        for k, v in c.get('profiles', {}).get('fast', {}).items():
            c[k].update(v) if isinstance(v, dict) else c.__setitem__(k, v)
    if smoke:
        c['experiment'].update(total_env_steps=1500, warmup_random_steps=300,
                               eval_every_steps=500, eval_episodes=2, seeds=[0])
        c['rollout'].update(num_rollouts=100, freq_steps=250)
    c['env']['backend'] = 'pybullet'
    c['env']['mjcf_scene'] = '/content/mujoco_menagerie/skydio_x2/scene.xml'
    return c

cfg = make_cfg(smoke=SMOKE, fast=True)
print('backend', cfg['env']['backend'], '| steps', cfg['experiment']['total_env_steps'],
      '| seeds', cfg['experiment']['seeds'])

In [ ]:
env, obs_dim, act_dim = drone_env.make_env(cfg['env'], seed=0, render=True)
obs0, _ = env.reset(seed=0)
print('obs_dim', obs_dim, '| act_dim', act_dim, '| HOVER_RPM', round(env.HOVER_RPM, 1))

## Environment samples (drone photos)

In [ ]:
env.reset(seed=0)
plots.render_frame(env, save_path=f'{DRIVE}/plots/env_frame.png');

In [ ]:
plots.render_filmstrip(env, policy='random', n_frames=6, steps_between=6, seed=1,
                       title='Random actions (drone tumbles)',
                       save_path=f'{DRIVE}/plots/env_random.png');

In [ ]:
plots.render_filmstrip(env, policy='hover', n_frames=6, steps_between=6, seed=1,
                       title='Zero thrust (drone drops)',
                       save_path=f'{DRIVE}/plots/env_zero.png');

## Train the four algorithms

In [ ]:
runs = []
for algo in cfg['experiment']['algorithms']:
    for seed in cfg['experiment']['seeds']:
        lp = f'{DRIVE}/logs/{algo}_seed{seed}.json'
        if os.path.exists(lp):
            print('skip (exists):', algo, seed); runs.append(json.load(open(lp)))
        else:
            runs.append(trainer.train_one(algo, cfg, DRIVE, seed=seed))
print('done:', len(runs), 'runs')

In [ ]:
plots.plot_sample_efficiency(runs, save_path=f'{DRIVE}/plots/train_sample_eff.png')
import pandas as pd
display(pd.DataFrame([{'algo': r['algo'], 'seed': r['seed'],
                       'best_return': round(r.get('best_return', 0.0), 1)} for r in runs]))